In [2]:
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from scipy.special import gamma

def fitness_function(subset, X_train, y_train, X_val, y_val, alpha=0.01):
    """
    Evaluates a binary feature subset.
    Fitness balances classification accuracy and penalty for number of selected features.
    """
    selected_indices = np.where(subset == 1)[0]

    if len(selected_indices) == 0:
        return 0.0

    X_tr = X_train[:, selected_indices]
    X_va = X_val[:, selected_indices]

    clf = RandomForestClassifier(n_estimators=50, random_state=42, n_jobs=-1)
    clf.fit(X_tr, y_train)
    y_pred = clf.predict(X_va)

    acc = accuracy_score(y_val, y_pred)

    num_selected = len(selected_indices)
    total_features = len(subset)
    fitness = acc - alpha * (num_selected / total_features)
    return fitness


def levy_flight(Lambda):
    """
    Generates Lévy flight step size using Mantegna's algorithm.
    """
    sigma_u = (gamma(1 + Lambda) * np.sin(np.pi * Lambda / 2) /
               (gamma((1 + Lambda) / 2) * Lambda * (2 ** ((Lambda - 1) / 2)))) ** (1 / Lambda)
    sigma_v = 1.0

    u = np.random.normal(0, sigma_u, size=1)
    v = np.random.normal(0, sigma_v, size=1)
    step = u / (abs(v) ** (1 / Lambda))
    return step[0]


def cuckoo_search_feature_selection(X_train, y_train, X_val, y_val,
                                  num_nests=15, max_iter=20, pa=0.25):
    num_features = X_train.shape[1]

    nests = np.random.randint(0, 2, size=(num_nests, num_features))
    for i in range(num_nests):
        if np.sum(nests[i]) == 0:
            nests[i, np.random.randint(num_features)] = 1

    fitness = np.array([fitness_function(nests[i], X_train, y_train, X_val, y_val)
                        for i in range(num_nests)])

    best_idx = np.argmax(fitness)
    best_nest = nests[best_idx].copy()
    best_fitness = fitness[best_idx]

    print(f"Initial Best Fitness: {best_fitness:.4f}")

    for iteration in range(max_iter):
        for i in range(num_nests):
            new_nest = nests[i].copy()
            step = levy_flight(1.5)
            for j in range(num_features):
                if np.random.rand() < abs(np.tanh(step)):
                    new_nest[j] = 1 - new_nest[j]  # Bit flip

            if np.sum(new_nest) == 0:
                new_nest[np.random.randint(num_features)] = 1

            new_fit = fitness_function(new_nest, X_train, y_train, X_val, y_val)

            j_rand = np.random.randint(num_nests)
            if new_fit > fitness[j_rand]:
                nests[j_rand] = new_nest
                fitness[j_rand] = new_fit

        for i in range(num_nests):
            if np.random.rand() < pa:
                nests[i] = np.random.randint(0, 2, size=num_features)
                if np.sum(nests[i]) == 0:
                    nests[i, np.random.randint(num_features)] = 1
                fitness[i] = fitness_function(nests[i], X_train, y_train, X_val, y_val)

        current_best_idx = np.argmax(fitness)
        if fitness[current_best_idx] > best_fitness:
            best_fitness = fitness[current_best_idx]
            best_nest = nests[current_best_idx].copy()

        print(f"Iteration {iteration + 1}/{max_iter} - Best Fitness: {best_fitness:.4f} - Features Selected: {np.sum(best_nest)}/{num_features}")

    return best_nest, best_fitness


if __name__ == "__main__":
    np.random.seed(42)
    X = np.random.rand(500, 41)
    y = np.random.randint(0, 2, size=500)

    X_train, X_val, y_train, y_val = train_test_split(X, y, test_size=0.3, random_state=42)

    print("Starting Cuckoo Search Feature Selection...")
    optimal_features_mask, best_score = cuckoo_search_feature_selection(
        X_train, y_train, X_val, y_val, num_nests=10, max_iter=10, pa=0.25
    )

    selected_feature_indices = np.where(optimal_features_mask == 1)[0]
    print("\nOptimization Complete!")
    print(f"Optimal Feature Indices: {selected_feature_indices}")
    print(f"Total Features Selected: {len(selected_feature_indices)} out of {X_train.shape[1]}")


Starting Cuckoo Search Feature Selection...
Initial Best Fitness: 0.4949
Iteration 1/10 - Best Fitness: 0.5361 - Features Selected: 16/41
Iteration 2/10 - Best Fitness: 0.5361 - Features Selected: 16/41
Iteration 3/10 - Best Fitness: 0.5361 - Features Selected: 16/41
Iteration 4/10 - Best Fitness: 0.5361 - Features Selected: 16/41
Iteration 5/10 - Best Fitness: 0.5361 - Features Selected: 16/41
Iteration 6/10 - Best Fitness: 0.5361 - Features Selected: 16/41
Iteration 7/10 - Best Fitness: 0.5361 - Features Selected: 16/41
Iteration 8/10 - Best Fitness: 0.5361 - Features Selected: 16/41
Iteration 9/10 - Best Fitness: 0.5361 - Features Selected: 16/41
Iteration 10/10 - Best Fitness: 0.5361 - Features Selected: 16/41

Optimization Complete!
Optimal Feature Indices: [ 0  2  4  6  8 10 12 15 16 18 25 31 32 36 37 39]
Total Features Selected: 16 out of 41


In [ ]:
!pip install -q pandas numpy scikit-learn matplotlib seaborn
